In [0]:

CREATE OR REPLACE TABLE claims.genai.hybrid_candidates AS
WITH gold_raw AS (
  SELECT explode(questions) AS q
  FROM read_files('/Volumes/claims/raw/landing/genai/gold_set.json', format => 'json', multiLine => 'true')
),
gold AS (
  SELECT q['id'] AS question_id, q['question'] AS question
  FROM gold_raw
  WHERE q['id'] IS NOT NULL AND q['id'] <> 'G19'
),
vec AS (
  SELECT g.question_id, vs.chunk_id, vs.search_score,
         row_number() OVER (PARTITION BY g.question_id ORDER BY vs.search_score DESC) AS vec_rank
  FROM gold g, LATERAL vector_search(
    index      => 'claims.genai.policy_chunks_idx',
    query_text => g.question,
    num_results => 20
  ) vs
),
kw_tokens AS (
  SELECT DISTINCT lower(token) AS token
  FROM gold g
  LATERAL VIEW explode(split(regexp_replace(g.question, '[^a-zA-Z0-9 ]+', ' '), ' ')) tk AS token
  WHERE lower(token) NOT IN ('the','a','an','is','are','on','of','to','in','for','by','and','or',
                             'what','which','how','does','do','must','be','there','with','from',
                             'bought','can','it','its','at','as','that','this','plan','plans')
    AND length(token) >= 3
),
kw AS (
  SELECT g.question_id, c.chunk_id,
         sum((length(lower(c.text)) - length(replace(lower(c.text), t.token, ''))) / length(t.token)) AS kw_score
  FROM gold g
  CROSS JOIN claims.genai.policy_chunks c
  JOIN kw_tokens t ON instr(lower(c.text), t.token) > 0
  GROUP BY g.question_id, c.chunk_id
),
kw_ranked AS (
  SELECT question_id, chunk_id, kw_score,
         row_number() OVER (PARTITION BY question_id ORDER BY kw_score DESC) AS kw_rank
  FROM kw
),
-- HYBRID fusion: reciprocal rank fusion of the vector ranking and the keyword ranking
hybrid AS (
  SELECT COALESCE(v.question_id, k.question_id)                    AS question_id,
         COALESCE(v.chunk_id, k.chunk_id)                          AS chunk_id,
         COALESCE(1.0 / (60 + v.vec_rank), 0)
           + COALESCE(1.0 / (60 + k.kw_rank), 0)                   AS fused_score
  FROM vec v
  FULL OUTER JOIN kw_ranked k
    ON v.question_id = k.question_id AND v.chunk_id = k.chunk_id
),
ranked AS (
  SELECT h.question_id, h.chunk_id, h.fused_score,
         v.search_score AS vec_score, k.kw_score,
         row_number() OVER (PARTITION BY h.question_id ORDER BY h.fused_score DESC) AS fused_rank
  FROM hybrid h
  LEFT JOIN vec v       ON v.question_id = h.question_id AND v.chunk_id = h.chunk_id
  LEFT JOIN kw_ranked k ON k.question_id = h.question_id AND k.chunk_id = h.chunk_id
)
SELECT question_id, chunk_id, fused_score, vec_score, kw_score, fused_rank
FROM ranked
WHERE fused_rank <= 10;

SELECT question_id, count(*) AS candidates
FROM claims.genai.hybrid_candidates
GROUP BY question_id ORDER BY question_id;

In [0]:
-- R4 Step 2: RERANK — LLM pointwise relevance scoring of the hybrid candidates.
-- Reranker used: databricks-meta-llama-3-1-8b-instruct served by ai_query, scoring each
-- candidate section 0-10 for how well it answers the question (see README.md).
-- Final order = rerank_score DESC, hybrid fused score as tie-breaker; top 5, best first.
CREATE OR REPLACE TABLE claims.genai.gold_answers AS
WITH gold_raw AS (
  SELECT explode(questions) AS q
  FROM read_files('/Volumes/claims/raw/landing/genai/gold_set.json', format => 'json', multiLine => 'true')
),
gold AS (
  SELECT q['id'] AS question_id, q['question'] AS question
  FROM gold_raw
  WHERE q['id'] IS NOT NULL AND q['id'] <> 'G19'
),
rerank AS (
  SELECT
    c.question_id,
    c.chunk_id,
    c.fused_score AS hybrid_fused_score,
    try_cast(regexp_extract(ai_query(
      'databricks-meta-llama-3-1-8b-instruct',
      concat('You are a search relevance judge. Rate from 0 to 10 how well the policy section answers the question. ',
             'Reply with ONLY a single integer 0-10 and nothing else.\n\nQuestion: ', g.question,
             '\n\nSection heading: ', c2.heading,
             '\n\nSection text: ', substr(c2.text, 1, 1200))
    ), '[0-9]+', 0) AS INT) AS rerank_score
  FROM claims.genai.hybrid_candidates c
  JOIN gold g ON g.question_id = c.question_id
  JOIN claims.genai.policy_chunks c2 ON c2.chunk_id = c.chunk_id
),
final AS (
  SELECT
    r.question_id,
    g.question,
    r.chunk_id,
    c2.doc_name,
    c2.heading,
    r.hybrid_fused_score,
    COALESCE(r.rerank_score, 0) AS rerank_score,
    'ai_query pointwise LLM reranker (databricks-meta-llama-3-1-8b-instruct, 0-10 relevance score)'
      AS reranker,
    row_number() OVER (
      PARTITION BY r.question_id
      ORDER BY COALESCE(r.rerank_score, 0) DESC, r.hybrid_fused_score DESC
    ) AS answer_rank
  FROM rerank r
  JOIN gold g ON g.question_id = r.question_id
  JOIN claims.genai.policy_chunks c2 ON c2.chunk_id = r.chunk_id
)
SELECT question_id, question, answer_rank, chunk_id, doc_name, heading,
       hybrid_fused_score, rerank_score, reranker
FROM final
WHERE answer_rank <= 5;

In [0]:
-- R4 Step 3: search-quality evaluation against the gold set.
-- hit-rate@5 and MRR over G01-G18 (G19 is excluded from the denominator by its own note).
-- Plus the client's ordering rule: G04/G05 must cite the individual-plans doc first,
-- G07/G08 must cite the group-plans doc first.
WITH gold_raw AS (
  SELECT explode(questions) AS q
  FROM read_files('/Volumes/claims/raw/landing/genai/gold_set.json', format => 'json', multiLine => 'true')
),
gold AS (
  SELECT q['id'] AS question_id, explode(from_json(q['expected_chunks'], 'array<string>')) AS expected_chunk
  FROM gold_raw
  WHERE q['id'] IS NOT NULL AND q['id'] <> 'G19' AND q['expected_chunks'] IS NOT NULL
),
hits AS (
  SELECT a.question_id,
         count(DISTINCT a.chunk_id)                       AS n_returned,
         count(DISTINCT e.expected_chunk)                 AS n_expected_found,
         min(CASE WHEN e.expected_chunk IS NOT NULL THEN a.answer_rank END) AS best_rank
  FROM claims.genai.gold_answers a
  LEFT JOIN gold e ON e.question_id = a.question_id AND e.expected_chunk = a.chunk_id
  GROUP BY a.question_id
),
metrics AS (
  SELECT
    round(avg(CASE WHEN n_expected_found > 0 THEN 1.0 ELSE 0.0 END), 4) AS hit_rate_at_5,
    round(avg(CASE WHEN best_rank IS NOT NULL THEN 1.0 / best_rank ELSE 0.0 END), 4) AS mrr,
    count(*) AS n_questions
  FROM hits
),
first_cite AS (
  SELECT question_id, doc_name
  FROM claims.genai.gold_answers
  WHERE answer_rank = 1 AND question_id IN ('G04','G05','G07','G08')
),
ordering_check AS (
  SELECT question_id, doc_name AS first_cited_doc,
    CASE
      WHEN question_id IN ('G04','G05')
           AND doc_name = '02_Waiting_Periods_Individual_Plans.md' THEN 'PASS'
      WHEN question_id IN ('G07','G08')
           AND doc_name = '03_Waiting_Periods_Group_Plans.md'      THEN 'PASS'
      ELSE 'FAIL'
    END AS ordering_rule
  FROM first_cite
)
SELECT 'OVERALL' AS check_name, NULL AS question_id, NULL AS detail,
       cast(n_questions AS STRING) AS value FROM metrics
UNION ALL
SELECT 'hit_rate@5', NULL, 'expected chunk found in top-5', cast(hit_rate_at_5 AS STRING) FROM metrics
UNION ALL
SELECT 'MRR', NULL, 'mean reciprocal rank of first expected chunk', cast(mrr AS STRING) FROM metrics
UNION ALL
SELECT 'ordering_rule', question_id, 'first cited doc', first_cited_doc || ' -> ' || ordering_rule
FROM ordering_check
ORDER BY check_name, question_id;

In [0]:
-- R4 Step 4: the deliverable — top 5 sections per gold question, best first.
SELECT question_id, question, answer_rank, chunk_id, doc_name, heading,
       hybrid_fused_score, rerank_score
FROM claims.genai.gold_answers
ORDER BY question_id, answer_rank;